# 02 · TF-IDF cleaning thresholds and the shared label guide (Lecture 3)

**Questions:** does TF-IDF help memo cleaning? Can four labelers apply one category guide consistently?

Lecture 3 participation question 1: compare three TF-IDF thresholds with a case/spacing-only baseline and report
useful cues lost, noise removed and empty outputs. Question 2: everyone labels the same 100 transactions
independently, then reconciles.

Our results (Lecture 4 review): on 100 transactions (708 tokens), `max_df = .10` lost 13% of useful cues and
removed no noise; `min_df >= 2` removed 66% of noise but lost 46% of cues and left 16 memos empty; `TF-IDF >= .3`
removed 26% of noise and lost 23% of cues. **Decision: keep TF-IDF for model features only** (the TF-IDF +
logistic regression baseline in notebook 03), and mask IDs with explicit rules instead (`text_norm`).

Labels: our initial taxonomies ranged from 15 to 21 categories and were merged into the 20-category guide
(README). AI and web research informed 14 of the 100 disputes; a person assigned every final label.

> Runs on DataHub. Cells show aggregate tables and charts only; files with memo text (errors, examples, predictions) stay in `outputs/` and are never printed here or committed.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
CONFIG = os.environ.get("CAPSTONE_CONFIG", "config.yaml")

import pandas as pd
from IPython.display import Image, display
from src.config import load_config
from src.main import run
OUT = load_config(CONFIG)["paths"]["out_dir"]
pd.set_option("display.max_colwidth", 80)

## TF-IDF token removal vs the baseline

In [ ]:
m = run(CONFIG, "tfidf")
pd.read_csv(OUT / "tfidf_cleaning_study.csv")

Useful cues are tokens inside a pre-tag rule match or transaction-type words; noise is ID-like
tokens. To use hand-marked tokens instead, set `tfidf_study.cue_file` in `config.yaml`. Helpful and harmful
before/after examples are in `outputs/tfidf_cleaning_examples.json` (memo text, DataHub only).

## Agreement on the shared 100 memos

In [ ]:
m = run(CONFIG, "agreement")
agree = json.load(open(OUT / "agreement.json"))
{k: agree[k] for k in ["memos", "all_agree", "majority_agree", "fleiss_kappa"]}

In [ ]:
pd.Series(agree["pairwise"], name="pairwise agreement").to_frame()

In [ ]:
pd.read_csv(OUT / "agreement_by_category.csv", index_col=0)

Categories with low unanimous shares are where the rulebook needed precedence rules (transaction
type over merchant, credit/debit direction for cash-advance apps, gambling under Subscriptions & Entertainment).